# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', 'N/A')}: {getattr(metadata, 'description', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's inspect the available record sets in the Croissant metadata, display their `@id`, name, and a sample of their fields.

In [ ]:
record_sets = getattr(metadata, 'record_set', []) if hasattr(metadata, 'record_set') else getattr(metadata, 'recordSet', [])

if not record_sets:
    print('No record sets found in the dataset metadata.')
else:
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
        rs_name = getattr(rs, 'name', 'N/A')
        print(f'\nRecord set @id: {rs_id}')
        print(f'  Name: {rs_name}')
        # List fields
        fields = getattr(rs, 'field', []) if hasattr(rs, 'field') else getattr(rs, 'fields', [])
        if fields:
            print('  Fields:')
            for fld in fields:
                fld_id = getattr(fld, '@id', None) or getattr(fld, 'id', None)
                fld_name = getattr(fld, 'name', 'N/A')
                print(f'    @id: {fld_id} \t Name: {fld_name}')
        else:
            print('  No fields listed in this record set.')

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will demonstrate extraction by loading data from the primary record set. You can modify the variable `target_record_set_id` to use a different record set found in the previous step.

In [ ]:
# Identify record set and field IDs
# If you know the record set @id (e.g., 'cr:main'), use it here. If not, re-run the cell above to find available ids.
record_sets = getattr(metadata, 'record_set', []) if hasattr(metadata, 'record_set') else getattr(metadata, 'recordSet', [])
if not record_sets:
    print('No record sets available.')
else:
    # For most Croissant datasets, there's often one main record set. We'll select the first one for demonstration.
    target_record_set = record_sets[0]
    target_record_set_id = getattr(target_record_set, '@id', None)

    # List all record set IDs
    all_record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]
    print(f'Available record set @ids: {all_record_set_ids}')

    # Load all record sets into DataFrames for flexible access later
    dataframes = {}

    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        print(f'Loading record set {rs_id} ...')
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f'  - {len(df)} rows')
        if len(df.columns) > 0:
            print(f'  - Columns: {list(df.columns)}')

    # Display the first few records of the primary record set
    print(f'\nFirst five records for {target_record_set_id}:')
    display(dataframes[target_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Filter and normalize a numeric field from the main record set
import numpy as np

# Select a numeric field for analysis
df = dataframes[target_record_set_id]

# Attempt to select a probable numeric field
numeric_candidate_cols = [c for c in df.columns if df[c].dtype in [np.int64, np.float64] or np.issubdtype(df[c].dtype, np.number)]
if not numeric_candidate_cols:
    # Some datasets come with all fields as objects/strings; try conversion
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
        except Exception:
            pass
    numeric_candidate_cols = [c for c in df.columns if df[c].dtype in [np.int64, np.float64] or np.issubdtype(df[c].dtype, np.number)]

if numeric_candidate_cols:
    numeric_field = numeric_candidate_cols[0]
    print(f'Using numeric field: {numeric_field}')

    # Example threshold
    threshold = df[numeric_field].quantile(0.75) if not df[numeric_field].isnull().all() else 0
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f'Filtered records with {numeric_field} > {threshold}:')
    print(filtered_df[[numeric_field]].head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f'Normalized {numeric_field} for filtered records:')
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try a grouping field (e.g., the first string/categorical column not used above)
    group_fields = [c for c in df.columns if c != numeric_field and df[c].dtype == object]
    if group_fields:
        group_field = group_fields[0]
        print(f'Grouping by {group_field}')
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index(name=f'{numeric_field}_mean')
        print('Grouped mean:')
        print(grouped_df.head())
else:
    print('No numeric columns found for EDA. Please specify a numeric field or verify the dataset columns.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_candidate_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If a group exists, plot mean value per group (top 10 group labels)
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,4))
        order = grouped_df.sort_values(f'{numeric_field}_mean', ascending=False)[group_field][:10]
        plot_data = grouped_df[grouped_df[group_field].isin(order)]
        sns.barplot(data=plot_data, x=group_field, y=f'{numeric_field}_mean')
        plt.title(f'Mean {numeric_field} by {group_field} (top 10)')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric columns found to visualize.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated loading a Croissant dataset using `mlcroissant`, discovering available record sets, extracting tabular data by record set `@id`, and conducting exploratory analysis on a numeric field.
- The workflow can be adapted for other datasets by specifying different record set and field `@id`s.

Explore the Croissant metadata and documentation for deeper insight and more advanced workflow designs.